### ECS 174 Computer Vision Final Project Code

#### Adam Hetherwick's code

November 2026

Microsoft COCO (common objects in context) dataset using a subset with 10 animal classes with bounding box and label annotations

Goal: Train a multi head model to best predict bounding box region (MSE loss) and animal classification (cross entropy loss). 

To implement: 

1. Subset data
2. Example images, EDA
3. Create train test split (important for pretrained model)
4. Implement models (YOLOv8, CNN? R-CNN?)
5. Evaluate results (runtime, mean Average Precision)
6. Compare results to COCO DPM (deformable parts model)?

#### Load dataset

In [2]:
coco_data_loc = "C:/Users/cheif/OneDrive/Documents/coco_data"

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
import requests
from tqdm import tqdm
from pycocotools.coco import COCO

os.chdir("C:/Users/cheif/OneDrive/Documents/GitHub/ECS_174_Final_Project")

coco_data_loc = "C:/Users/cheif/OneDrive/Documents/coco_data"

def split_dir(split):
    return f'{coco_data_loc}/data/{split}2017'            # single source of truth for the path

def download_animal_images(split, max_workers=16):
    out_dir = split_dir(split)
    marker = os.path.join(out_dir, '.download_complete')

    if os.path.exists(marker):                            # only checks the split being requested
        print(f'{split}: already downloaded, skipping')
        return

    coco = COCO(f'{coco_data_loc}/annotations/instances_{split}2017.json')

    animal_ids = coco.getCatIds(supNms=['animal'])        # the 10 official animal classes
    img_ids = set()
    for cid in animal_ids:
        img_ids.update(coco.getImgIds(catIds=[cid]))
    imgs = coco.loadImgs(list(img_ids))
    print(f'{split}: {len(imgs)} animal images')

    os.makedirs(out_dir, exist_ok=True)

    def fetch(img):
        path = os.path.join(out_dir, img['file_name'])
        if os.path.exists(path):                          # skip files you already have
            return
        r = requests.get(img['coco_url'], timeout=30)
        r.raise_for_status()
        with open(path, 'wb') as f:
            f.write(r.content)

    with ThreadPoolExecutor(max_workers) as ex:
        list(tqdm(ex.map(fetch, imgs), total=len(imgs)))  # progress bar; errors still raise here

    open(marker, 'w').close()                             # only reached if every download succeeded
    print(f'{split}: done')

for split in ['val', 'train']:
    download_animal_images(split)

loading annotations into memory...
Done (t=0.53s)
creating index...
index created!
val: 1016 animal images


100%|██████████| 1016/1016 [00:00<00:00, 507190.30it/s]

val: done
loading annotations into memory...


Done (t=13.45s)
creating index...
index created!
train: 23989 animal images


100%|██████████| 23989/23989 [00:00<00:00, 142599.83it/s]


train: done


In [11]:
download_animal_images('val')     # ~1,000 images, quick

val: already downloaded, skipping


WARNING: Below cell may take 5-30 minutes to run depending on compute power.

Last run: ~14 min

In [12]:
download_animal_images('train')   # ~24,000 images, takes a while

train: already downloaded, skipping


Note: We use the COCO:validation set as our test set because COCO:test labels are private.

#### Example Image Viewing (train, test)

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

ROOT = f'{coco_data_loc}/data'

def load_split(split):

    coco = COCO(f'{ROOT}/annotations/instances_{split}2017.json')

    animal_ids = coco.getCatIds(supNms=['animal'])          # the 10 animal classes
    img_ids = set()
    for cid in animal_ids:
        img_ids.update(coco.getImgIds(catIds=[cid]))
    # keep only images that are actually downloaded
    img_ids = [i for i in img_ids
               if os.path.exists(f"{ROOT}/{split}2017/{coco.imgs[i]['file_name']}")]
    return coco, animal_ids, img_ids

def draw(ax, coco, animal_ids, split, img_id):
    info = coco.imgs[img_id]
    img = Image.open(f"{ROOT}/{split}2017/{info['file_name']}").convert('RGB')
    ax.imshow(img)
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id, catIds=animal_ids, iscrowd=None))
    for a in anns:
        x, y, w, h = a['bbox']                               # COCO format: [x_min, y_min, width, height]
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
        ax.text(x, y - 3, coco.cats[a['category_id']]['name'],
                color='black', fontsize=9, backgroundcolor='lime')
    ax.set_title(f"{split}: {info['file_name']}", fontsize=9)
    ax.axis('off')

In [14]:
splits = {s: load_split(s) for s in ['train', 'val']}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, split in enumerate(['train', 'val']):               # row 0 = train, row 1 = validation
    coco, animal_ids, img_ids = splits[split]
    for ax, img_id in zip(axes[row], random.sample(img_ids, 3)):
        draw(ax, coco, animal_ids, split, img_id)
plt.tight_layout()
plt.show()

loading annotations into memory...


FileNotFoundError: [Errno 2] No such file or directory: 'annotations/instances_train2017.json'